# 1. Initializations

## 1.1 General imports

In [ ]:
### global
import logging
import os
from typing import cast
from smartcheck.logger_config import setup_logger
setup_logger(logging.INFO)
print(f'Path [{os.environ["PATH"]}]')

# Test pytorch GPU config
import torch
cuda_test = torch.cuda.is_available()
print(f"✅ Torch CUDA available: {cuda_test}")
device_name = torch.cuda.get_device_name(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Device Name: {device_name} | Device reference: {device}")

### machine learning (scikit-learn)
import pandas as pd
import numpy as np
import torch
from sklearn.pipeline import Pipeline
from tsfm_public import (
    TimeSeriesForecastingPipeline,
    TinyTimeMixerForPrediction,
)
from tsfm_public.toolkit.visualization import plot_predictions

### graphical
import matplotlib.pyplot as plt
# for jupyter notebook management
%matplotlib inline


## 1.2 Project Specific imports

In [ ]:
import smartcheck.dataframe_common as dfc
import smartcheck.preprocessing_project_specific as pps
import smartcheck.deep_learning_project_specific as dlps
import smartcheck.modeling_project_specific as mps

# 2. Loading and Preprocessing

In [ ]:
df_cpt_raw = dfc.load_dataset_from_config('velo_comptage_ml_ready_data', sep=',', index_col=0)

if df_cpt_raw is not None and isinstance(df_cpt_raw, pd.DataFrame):
    df_cpt = df_cpt_raw.copy()

In [ ]:
df_cpt.info()

## 2.1 Preprocessing pipelines

In [ ]:
keep_cols = [
    "nom_du_site_de_comptage",
    "comptage_horaire",
    # "date_et_heure_de_comptage",
    "date_et_heure_de_comptage_local",
    # "date_et_heure_de_comptage_utc",
    "orientation_compteur",
    # "latitude",
    # "longitude",
    # "arrondissement",
    "jour_ferie",
    "vacances_scolaires",
    "temperature_2m_c",
    "rain_mm",
    "snowfall_cm",
    # "weather_code_wmo_code",
    # "elevation",
    "weather_code_wmo_code_category",
]

pipe_preproc = Pipeline([
    ("add_datetime_features", pps.DatetimePreprocessingTransformer(timestamp_col="date_et_heure_de_comptage",
                                                                   for_sarimax=True)),
    ("filter_columns", pps.ColumnFilterTransformer(columns_to_keep=keep_cols)),
])

df_preproc = pipe_preproc.fit_transform(df_cpt)
if df_preproc is not None and isinstance(df_preproc, pd.DataFrame):
    df = df_preproc.copy()

In [ ]:
# Verification des distributions après preprocessing
df.info()
display(df.select_dtypes(include=np.number).describe())
display(df.select_dtypes(include='object').describe())

# 3. Regression modeling

## 3.1 Variables de contexte

In [ ]:
dict_compteurs = {
    "experiment_1": {
        "key": ('135 avenue Daumesnil','SE-NO'),
        "name": "Daumesnil S-N",
        "sub_range": (0,),
    },
    "experiment_2": {
        "key": ('102 boulevard de Magenta', 'SE-NO'),
        "name": "Magenta-O-E",
        "sub_range": (0,),
    },
    "experiment_3": {
        "key": ('Totem 73 boulevard de Sébastopol', 'S-N'),
        "name": "Sébastopol_S-N",
        "sub_range": (0,),
    },
}
timestamp_column = "date_et_heure_de_comptage_local"
target_columns = ["comptage_horaire"]
context_length = 512 # nombre de variable explicatives maximale (non utilisé ici, 512 est le maximum permis par le modèle)
prediction_length = 96 # nombre de prédictions à prévoir (96 est le maximum permis par le modèle tiny)
grouped_df = df.groupby(["nom_du_site_de_comptage", "orientation_compteur"])

## 3.2 Visualisation de contexte

In [ ]:
for experiment, exp_params in dict_compteurs.items():
    key = exp_params["key"]
    if key in grouped_df.groups:
        df_compteur = grouped_df.get_group(key)
        logging.info(f"\n--- {experiment} : {exp_params} ---")
    else:
        logging.info(f"⚠️ Clé {key} non trouvée dans les groupes de df.")
        continue

    df_compteur = df_compteur.sort_values(by=timestamp_column)
    sub_range = exp_params["sub_range"]
    range_start = sub_range[0]
    range_end = sub_range[1] if len(sub_range) > 1 else None
    df_compteur_sub = df_compteur[range_start:range_end]

    fig, axs = plt.subplots(len(target_columns), 1, figsize=(10, 2 * len(target_columns)), squeeze=False)
    for ax, target_column in zip(axs, target_columns):
        ax[0].plot(df_compteur_sub[timestamp_column], df_compteur_sub[target_column])
    plt.show()

## 3.3 Transfert learning avec preprocessing et prediction

#### Utilisation du modèle granite en zero shot sur nos données

In [ ]:
zeroshot_model = TinyTimeMixerForPrediction.from_pretrained(
    "ibm-granite/granite-timeseries-ttm-r2",  # Name of the model on Hugging Face
    num_input_channels=len(target_columns),  # tsp.num_input_channels
)

In [ ]:
pipeline = TimeSeriesForecastingPipeline(
    zeroshot_model,
    timestamp_column=timestamp_column,
    id_columns=[],
    target_columns=target_columns,
    explode_forecasts=False,
    prediction_length=prediction_length,
    freq="h",
    device=device,  # Specify your local GPU or CPU.
)

forecast_results = {}
# Make a forecast on the target column given the input data.
for experiment, exp_params in dict_compteurs.items():
    key = exp_params["key"]
    if key in grouped_df.groups:
        df_compteur = grouped_df.get_group(key)
        logging.info(f"\n--- {experiment} : {exp_params} ---")
    else:
        logging.info(f"⚠️ Clé {key} non trouvée dans les groupes de df.")
        continue

    df_compteur = df_compteur.sort_values(by=timestamp_column)
    sub_range = exp_params["sub_range"]
    range_start = sub_range[0]
    range_end = sub_range[1] if len(sub_range) > 1 else None
    df_compteur_sub = df_compteur[range_start:range_end]

    df_train, df_test = dlps.df_train_test_split_time_aware(
        df_compteur_sub,
        timestamp_column=timestamp_column,
        test_size=0.2,
        sort=True,
    )

    # Calcul de Prediction train et test
    predictions_df_train = cast(pd.DataFrame,pipeline(df_train))
    predictions_df_test = cast(pd.DataFrame,pipeline(df_test))

    #############################################
    ### Calcul des données pour les métriques ###
    #############################################
    y_train = predictions_df_train.comptage_horaire.apply(
        lambda x: x[0]
    ).iloc[:-1]
    y_train_pred = predictions_df_train.comptage_horaire_prediction.apply(
        lambda x: x[0]
    ).iloc[:-1]
    y_test = predictions_df_test.comptage_horaire.apply(
        lambda x: x[0]
    ).iloc[:-1]
    y_test_pred = predictions_df_test.comptage_horaire_prediction.apply(
        lambda x: x[0]
    ).iloc[:-1]
    dates_test = predictions_df_test[["date_et_heure_de_comptage_local"]].iloc[:-1]
    forecast_results[experiment] = {
        "exp_params":exp_params,
        "predictions_df_train":predictions_df_train,
        "predictions_df_test":predictions_df_test,
        "df_train":df_train,
        "df_test":df_test,
        "y_train":y_train,
        "y_train_pred":y_train_pred,
        "y_test":y_test,
        "y_test_pred":y_test_pred,
        "dates_test":dates_test,
    }


In [ ]:
for experiment, forecast_result in forecast_results.items():
    exp_params = forecast_result["exp_params"]
    compteur_key = exp_params["key"]
    logging.info(f"\n\nParamètres {exp_params} :\n")
    df_test = forecast_result["df_test"]  # type: ignore
    predictions_df_test = forecast_result["predictions_df_test"]  # type: ignore
    df_compteur = df_compteur.sort_values(by=timestamp_column)
    sub_range = exp_params["sub_range"]
    range_start = sub_range[0]
    range_end = sub_range[1] if len(sub_range) > 1 else None
    df_compteur_sub = df_compteur[range_start:range_end]
    plot_predictions(
        input_df=df_test,
        predictions_df=predictions_df_test,
        freq="h",
        timestamp_column=timestamp_column,
        channel=target_columns[0],
        # prediction en partant du dernier échantillon connu (-1) 
        # et aléatoirement à -168 lags (1 semaine avant), -336 lags (2 semaines avant)
        indices=[-1, -168, -336],  
    )
    plt.show()

    y_train = forecast_result["y_train"]
    y_train_pred = forecast_result["y_train_pred"]
    y_test = forecast_result["y_test"]
    y_test_pred = forecast_result["y_test_pred"]
    dates_test = forecast_result["dates_test"]
    periode_limite = (
        dates_test.date_et_heure_de_comptage_local[len(dates_test)-24*7*2],
        dates_test.date_et_heure_de_comptage_local[len(dates_test)-1], # 5 dernière semaines
    )

    # Affichage des metrique train et test
    model_train_metrics = mps.compute_metrics(
        y_train,
        y_train_pred,
    )
    model_test_metrics = mps.compute_metrics(
        y_test,
        y_test_pred
    )
    logging.info(f"Metriques du modèle (Train): {model_train_metrics}")
    logging.info(f"Metriques du modèle (Test): {model_test_metrics}")

    # projection des predictions de test dans le temps
    fig_pred = mps.plot_predictions(
        str(key),
        dates_test, 
        y_test, 
        y_test_pred, 
        periode_limite=periode_limite,
    )
    plt.show()

    # projection des résidus et calcul du coefficient de dérive dans le temps
    fig1_res, fig2_res, model_res_coeff = mps.compute_residuals_plot(
        str(key),
        dates_test, 
        y_test, 
        y_test_pred.values, 
        periode_limite=periode_limite
    )
    logging.info(f"Pente de la droite de régression des résidus dans le temps (dérive) : {model_res_coeff}")
    plt.show()